# Multi-Plant Production Planning: Minimizing Supply-Chain Cost with PROC OPTMODEL (MILP)

## Executive summaryA discrete-goods manufacturer must decide which of three plants to run aproduction line at, and how many units to ship from each open plant to fourregional distribution centers (DCs), so that firm demand is met at the lowesttotal monthly cost. The trade-off is between a plant's fixed line-activationcost and the production-plus-freight savings of routing demand through it.We formulate the decision as a mixed-integer linear program (MILP) in`PROC OPTMODEL` and solve it with the MILP solver. For this instance theoptimizer **opens plants P1 and P2 and keeps P3 closed**, for a minimum totalmonthly cost of **76,990**. Plant P3 stays dark because its 7,000 fixed costis not recovered by the modest production and freight savings it would offeronce the cheaper, higher-capacity plants P1 and P2 already cover all demand.We then run a demand-sensitivity analysis by re-solving the MILP with eachDC's demand raised by 100 units. The marginal cost of an extra 100 units ishighest at **DC1 (+3,000)** and lowest at **DC4 (+2,600)**, identifying DC1 asthe most expensive lane to serve at the margin.

## Data sourcesAll data are generated synthetically inside the notebook (small hand-settables) so the example is fully reproducible with no external files or networkaccess.| Dataset | Rows | Key columns | Description ||---------|------|-------------|-------------|| `plants` | 3 | `plant`, `line_fixed`, `capacity`, `prod_cost` | Per-plant fixed cost to switch on a production line, the line's monthly unit capacity, and the per-unit variable production cost. || `demand` | 4 | `dc`, `units` | Firm monthly demand (units) at each regional distribution center. || `lanes` | 12 | `plant`, `dc`, `freight` | One row per plant-to-DC lane: the per-unit freight cost to move a unit from the plant to the DC. |Total demand is 2,080 units; total capacity across all three plants is 3,600units, so the network has slack and the open/close decision is a genuineeconomic choice rather than one forced by feasibility.

## Step 1 - Build the synthetic supply-chain tablesWe build three small tables that mirror a planning extract: `plants`(fixed cost, capacity, unit production cost), `demand` (firm unit demand perDC), and `lanes` (per-unit freight on every plant -> DC lane). Plant P2 is thelarge, low-variable-cost plant; P3 is small with the highest production costbut the lowest fixed cost. Freight rises with distance, so each plant ischeapest for the DCs nearest it.

In [1]:
data plants;
    length plant $2;
    input plant $ line_fixed capacity prod_cost;
    datalines;
P1 9000 1200 22
P2 12000 1500 20
P3 7000 900 27
;
run;

data demand;
    length dc $3;
    input dc $ units;
    datalines;
DC1 440
DC2 540
DC3 570
DC4 530
;
run;

data lanes;
    length plant $2 dc $3;
    input plant $ dc $ freight;
    datalines;
P1 DC1 8
P1 DC2 6
P1 DC3 9
P1 DC4 12
P2 DC1 11
P2 DC2 7
P2 DC3 6
P2 DC4 5
P3 DC1 14
P3 DC2 12
P3 DC3 10
P3 DC4 9
;
run;

proc print data=plants noobs; title 'Plants: fixed line cost, capacity, unit production cost'; run;
proc print data=demand noobs; title 'Distribution-center demand'; run;
proc print data=lanes  noobs; title 'Plant -> DC lanes: freight cost per unit'; run;
title;

                                Plants: fixed line cost, capacity, unit production cost                                 

PLANT  LINE_FIXED  CAPACITY  PROD_COST
P1           9000      1200         22
P2          12000      1500         20
P3           7000       900         27

                                               Distribution-center demand                                               

 DC  UNITS
DC1    440
DC2    540
DC3    570
DC4    530

                                        Plant -> DC lanes: freight cost per unit                                        

PLANT   DC  FREIGHT
P1     DC1        8
P1     DC2        6
P1     DC3        9
P1     DC4       12
P2     DC1       11
P2     DC2        7
P2     DC3        6
P2     DC4        5
P3     DC1       14
P3     DC2       12
P3     DC3       10
P3     DC4        9

NOTE: DATA plants

NOTE: Processing inline DATALINES (3 lines)

NOTE: Read 3 rows from DATALINES.
NOTE: Wrote plants (3 rows, 4 columns).
NOTE: DATA elapsed:
  

## Step 2 - Formulate and solve the MILPWe read the three tables into `PROC OPTMODEL` sets and parameters, then declarethe decision variables:- `Open[p]` - **binary**: 1 if plant `p` runs a line this month, 0 otherwise.- `Ship[p,d]` - **continuous, non-negative**: units shipped from plant `p` to DC `d`.The **objective** minimizes total monthly cost: line-activation fixed cost plusproduction-plus-freight cost on every lane. We write it as a single outer sumover plants, with each plant's fixed cost and its outbound shipping cost foldedtogether, so the fixed and variable contributions are grouped per plant.**Constraints:**- `MeetDemand[d]` - every DC's demand is met exactly.- `LineCapacity[p]` - a plant ships no more than its capacity, **and only if its  line is open** (the linking constraint `sum_d Ship[p,d] <= capacity[p]*Open[p]`).We solve with the MILP solver.

In [2]:
proc optmodel;
    /* ---- Index sets ---- */
    set<str> PLANTS;
    set<str> DCS;

    /* ---- Parameters read from the synthetic tables ---- */
    num line_fixed{PLANTS};
    num capacity{PLANTS};
    num prod_cost{PLANTS};
    num dmd{DCS};
    num freight{PLANTS, DCS};

    read data plants into PLANTS=[plant] line_fixed capacity prod_cost;
    read data demand into DCS=[dc] dmd=units;
    read data lanes  into [plant dc] freight;

    /* ---- Decision variables ---- */
    var Open{PLANTS} binary;
    var Ship{PLANTS, DCS} >= 0;

    /* ---- Objective: total monthly supply-chain cost ----
       One outer sum over plants; each plant contributes its fixed
       line cost (if open) plus the production + freight on its lanes. */
    min TotalCost =
        sum{p in PLANTS} (
            line_fixed[p] * Open[p]
            + sum{d in DCS} (prod_cost[p] + freight[p,d]) * Ship[p,d]
        );

    /* ---- Meet every DC's demand exactly ---- */
    con MeetDemand{d in DCS}:
        sum{p in PLANTS} Ship[p,d] = dmd[d];

    /* ---- Capacity is available only when the line is open ---- */
    con LineCapacity{p in PLANTS}:
        sum{d in DCS} Ship[p,d] <= capacity[p] * Open[p];

    solve with milp;

    print Open;
    print Ship;
quit;


                        The OPTMODEL Procedure

                           Problem Summary
  Objective Sense               Minimization
  Objective Function            TOTALCOST
  Objective Type                Linear

  Number of Variables           15
  Bounded Above                 0
  Bounded Below                 12
  Bounded Below and Above       3
  Free                          0
  Fixed                         0

  Number of Constraints         7
  Integer Variables             3

                          Solution Summary
  Solver                        MILP
  Objective Function            TOTALCOST
  Solution Status               Optimal
  Objective Value               76990.0000000000

           i            OPEN

          P1    1.0000000000
          P2    1.0000000000
          P3    0.0000000000

           i            SHIP

      P1,DC1  440.0000000000
      P1,DC2  140.0000000000
      P1,DC3    0.0000000000
      P1,DC4    0.0000000000
      P2,DC1    0.0000000000


## Step 3 - Persist the production planA plan is only useful if the rest of the business can consume it. We re-stateand re-solve the model, then use `CREATE DATA` to write a plant-levelopen/closed summary that finance and operations can act on. The shippingvolumes themselves are printed above in the `Ship` listing (every lane with apositive value is a shipment to execute).

In [3]:
proc optmodel;
    set<str> PLANTS;
    set<str> DCS;
    num line_fixed{PLANTS};
    num capacity{PLANTS};
    num prod_cost{PLANTS};
    num dmd{DCS};
    num freight{PLANTS, DCS};

    read data plants into PLANTS=[plant] line_fixed capacity prod_cost;
    read data demand into DCS=[dc] dmd=units;
    read data lanes  into [plant dc] freight;

    var Open{PLANTS} binary;
    var Ship{PLANTS, DCS} >= 0;

    min TotalCost =
        sum{p in PLANTS} (
            line_fixed[p] * Open[p]
            + sum{d in DCS} (prod_cost[p] + freight[p,d]) * Ship[p,d]
        );

    con MeetDemand{d in DCS}:
        sum{p in PLANTS} Ship[p,d] = dmd[d];
    con LineCapacity{p in PLANTS}:
        sum{d in DCS} Ship[p,d] <= capacity[p] * Open[p];

    solve with milp;

    /* Plant-level open/closed summary with capacity and fixed cost. */
    create data plant_summary from
        [plant]
        is_open    = Open[plant]
        capacity   = capacity[plant]
        line_fixed = line_fixed[plant];
quit;

proc print data=plant_summary noobs;
    title 'Plant activation decision (is_open = 1 when the line runs)';
run;
title;


                        The OPTMODEL Procedure

                           Problem Summary
  Objective Sense               Minimization
  Objective Function            TOTALCOST
  Objective Type                Linear

  Number of Variables           15
  Bounded Above                 0
  Bounded Below                 12
  Bounded Below and Above       3
  Free                          0
  Fixed                         0

  Number of Constraints         7
  Integer Variables             3

                          Solution Summary
  Solver                        MILP
  Objective Function            TOTALCOST
  Solution Status               Optimal
  Objective Value               76990.0000000000

                               Plant activation decision (is_open = 1 when the line runs)                               

PLANT  IS_OPEN  CAPACITY  LINE_FIXED
P1           1      1200        9000
P2           1      1500       12000
P3           0       900        7000

NOTE: PROC OPTMODEL 



## Step 4 - Demand sensitivity by re-solvingTo ask *"how much would total cost rise if one more batch of demand appeared ata given DC?"* we re-solve the full MILP four times, each time raising one DC'sdemand by 100 units while holding everything else fixed, and compare the newoptimum against the baseline of **76,990**.The increase in the optimal objective is the **marginal cost of serving 100more units at that DC** - a true what-if from the optimizer, accounting forwhich plant picks up the extra load and any capacity it bumps into. DCs withthe largest increase are where the network is most strained and where extracapacity, a closer plant, or freight negotiation would pay off first.

In [4]:
/* Build one demand table per scenario: each raises a single DC by 100. */
data demand_dc1; set demand; if dc = 'DC1' then units = units + 100; run;
data demand_dc2; set demand; if dc = 'DC2' then units = units + 100; run;
data demand_dc3; set demand; if dc = 'DC3' then units = units + 100; run;
data demand_dc4; set demand; if dc = 'DC4' then units = units + 100; run;

/* Re-solve the MILP once per scenario; the Solution Summary's
   Objective Value is that scenario's minimum total cost. */
%macro scenario(dsn=);
proc optmodel;
    set<str> PLANTS;
    set<str> DCS;
    num line_fixed{PLANTS};
    num capacity{PLANTS};
    num prod_cost{PLANTS};
    num dmd{DCS};
    num freight{PLANTS, DCS};

    read data plants into PLANTS=[plant] line_fixed capacity prod_cost;
    read data &dsn   into DCS=[dc] dmd=units;
    read data lanes  into [plant dc] freight;

    var Open{PLANTS} binary;
    var Ship{PLANTS, DCS} >= 0;

    min TotalCost =
        sum{p in PLANTS} (
            line_fixed[p] * Open[p]
            + sum{d in DCS} (prod_cost[p] + freight[p,d]) * Ship[p,d]
        );

    con MeetDemand{d in DCS}:
        sum{p in PLANTS} Ship[p,d] = dmd[d];
    con LineCapacity{p in PLANTS}:
        sum{d in DCS} Ship[p,d] <= capacity[p] * Open[p];

    solve with milp;
quit;
%mend scenario;

title 'Scenario: DC1 demand + 100'; %scenario(dsn=demand_dc1);
title 'Scenario: DC2 demand + 100'; %scenario(dsn=demand_dc2);
title 'Scenario: DC3 demand + 100'; %scenario(dsn=demand_dc3);
title 'Scenario: DC4 demand + 100'; %scenario(dsn=demand_dc4);
title;

                                               Scenario: DC1 demand + 100                                               


                        The OPTMODEL Procedure

                           Problem Summary
  Objective Sense               Minimization
  Objective Function            TOTALCOST
  Objective Type                Linear

  Number of Variables           15
  Bounded Above                 0
  Bounded Below                 12
  Bounded Below and Above       3
  Free                          0
  Fixed                         0

  Number of Constraints         7
  Integer Variables             3

                          Solution Summary
  Solver                        MILP
  Objective Function            TOTALCOST
  Solution Status               Optimal
  Objective Value               79990.0000000000

                                               Scenario: DC2 demand + 100                                               


                        The OPTMODEL Procedure



## Interpreting the results- **The plan.** The MILP opens **P1 and P2** and keeps **P3 closed**, at a  minimum total monthly cost of **76,990**. The `Ship` listing in Step 2 gives  the exact lane volumes: P1 supplies DC1 (440) and most of DC2 (140), while P2  supplies the rest of DC2 (400) plus all of DC3 (570) and DC4 (530). Both open  plants run well inside capacity (P1 ships 580 of 1,200; P2 ships 1,500 of  1,500 - P2 is at its cap). P3 stays dark because its 7,000 fixed cost is not  recovered: P1 and P2 together can cover all 2,080 units, and re-solving with  P3 forced open costs 83,990 - exactly 7,000 more, confirming the line earns  nothing once the cheaper plants are running.- **Why MILP and not just LP.** The `Open[p]` binaries and the linking  constraint `Ship <= capacity*Open` make the decision discrete: you cannot ship  from a closed plant, and switching a line on is a lumpy, all-or-nothing cost.  An LP relaxation could open a plant "fractionally," which has no operational  meaning; the integer decision is what tells us, concretely, to leave P3 off.- **Demand sensitivity.** Raising each DC's demand by 100 units gives marginal  costs of **DC1 +3,000, DC2 +2,800, DC3 +2,700, DC4 +2,600** (the Step 4  objectives 79,990 / 79,790 / 79,690 / 79,590 versus the 76,990 baseline).  Per unit that is 30, 28, 27, and 26 respectively, so **DC1 is the most  expensive DC to serve at the margin and DC4 the cheapest** - DC1 is the first  place to look for a closer plant or a freight discount, while extra demand at  DC4 is comparatively cheap to absorb.- **Where to take it next.** Natural extensions are multi-period planning with  inventory carryover, minimum-run quantities (another set of binaries),  service-level constraints on single-sourcing, or a wider scenario sweep over  demand uncertainty. `PROC OPTMODEL` expresses all of these with the same  set / parameter / var / con vocabulary used here.